<a href="https://colab.research.google.com/github/sandeep123-R/cat-vs-dog/blob/main/cat_vs_dogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anthonytherrien/dog-vs-cat")

print("Path to dataset files:", path)

100%|██████████| 360M/360M [00:06<00:00, 54.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/anthonytherrien/dog-vs-cat/versions/3


In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.mobilenet import preprocess_input


# =========================
# 2. DOWNLOAD DATASET
# =========================
import kagglehub

# Original download path (read-only)
read_only_path = kagglehub.dataset_download("anthonytherrien/dog-vs-cat")
print("Dataset read-only path:", read_only_path)

print("Folders inside read-only path:", os.listdir(read_only_path))


# =========================
# 3. FIX DATASET STRUCTURE (VERY IMPORTANT)
#    - Create a writable directory for reorganized data
# =========================

# Define a writable path for our reorganized dataset
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

animals_path = os.path.join(read_only_path, "animals")

# Create cats and dogs folders within the writable data_dir
cats_path = os.path.join(data_dir, "cats")
dogs_path = os.path.join(data_dir, "dogs")

os.makedirs(cats_path, exist_ok=True)
os.makedirs(dogs_path, exist_ok=True)

# Copy files from the read-only source to the writable destination
# Iterate through subfolders like 'dog' and 'cat' within animals_path
for animal_type_folder in os.listdir(animals_path):
    current_animal_type_path = os.path.join(animals_path, animal_type_folder)
    if os.path.isdir(current_animal_type_path): # Ensure it's a directory
        for file_name in os.listdir(current_animal_type_path):
            src = os.path.join(current_animal_type_path, file_name)
            if "cat" in animal_type_folder.lower():
                shutil.copy(src, cats_path)
            elif "dog" in animal_type_folder.lower():
                shutil.copy(src, dogs_path)

print("Dataset organized into cats & dogs folders in a writable location!")


# =========================
# 4. LOAD DATASET
#    - Use the new writable data_dir
# =========================
img_size = (224, 224)
batch_size = 32

train_ds = image_dataset_from_directory(
    data_dir, # Use the writable directory
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    data_dir, # Use the writable directory
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)


# =========================
# 5. CHECK CLASS NAMES (CRITICAL)
# =========================
print("Class names:", train_ds.class_names)


# =========================
# 6. MOBILE-NET PREPROCESSING
# =========================
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))


# =========================
# 7. DATA AUGMENTATION (ONLY TRAIN)
# =========================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))


# =========================
# 8. OPTIMIZE PERFORMANCE
# =========================
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


# =========================
# 9. VERIFY PREPROCESSING
# =========================
for images, labels in train_ds.take(1):
    print("Pixel range:", images[0].numpy().min(), images[0].numpy().max())

Using Colab cache for faster access to the 'dog-vs-cat' dataset.
Dataset read-only path: /kaggle/input/dog-vs-cat
Folders inside read-only path: ['animals']
Dataset organized into cats & dogs folders in a writable location!
Found 1000 files belonging to 2 classes.
Using 800 files for training.
Found 1000 files belonging to 2 classes.
Using 200 files for validation.
Class names: ['cats', 'dogs']
Pixel range: -1.0 0.9188981


In [ ]:
base_model = tf.keras.applications.MobileNet(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)

output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

new_model = tf.keras.Model(inputs=base_model.input, outputs=output)

# =========================
# 12. COMPILE MODEL
# =========================
new_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# =========================
# 13. MODEL SUMMARY
# =========================
new_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 112, 112, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 112, 112, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (ReLU)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1 (DepthwiseConv2D)     │ (None, 112, 112, 32)   │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_bn                    │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_relu (ReLU)           │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1 (Conv2D)              │ (None, 112, 112, 64)   │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_bn                    │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_relu (ReLU)           │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pad_2 (ZeroPadding2D)      │ (None, 113, 113, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2 (DepthwiseConv2D)     │ (None, 56, 56, 64)     │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_bn                    │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_relu (ReLU)           │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2 (Conv2D)              │ (None, 56, 56, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_relu (ReLU)           │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3 (DepthwiseConv2D)     │ (None, 56, 56, 128)    │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_relu (ReLU)           │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3 (Conv2D)              │ (None, 56, 56, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 3,364,289 (12.83 MB)

 Trainable params: 133,377 (521.00 KB)

 Non-trainable params: 3,230,912 (12.32 MB)

In [ ]:
# =========================
# 14. CALLBACKS
# =========================
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]


# =========================
# 15. TRAIN MODEL
# =========================
history =new_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 14s 297ms/step - accuracy: 0.8450 - loss: 0.3826 - val_accuracy: 0.9850 - val_loss: 0.1452
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9775 - loss: 0.0913 - val_accuracy: 0.9900 - val_loss: 0.0671
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9912 - loss: 0.0443 - val_accuracy: 0.9900 - val_loss: 0.0425
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9962 - loss: 0.0256 - val_accuracy: 0.9900 - val_loss: 0.0301
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9950 - loss: 0.0249 - val_accuracy: 0.9950 - val_loss: 0.0218
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9962 - loss: 0.0199 - val_accuracy: 0.9950 - val_loss: 0.0170
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 1.0000 - loss: 0.0116 - val_accuracy: 0.9950 - val_loss: 0.0138
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 1.0000 - loss: 0.0112 - val_accuracy: 0.9950 -

In [ ]:
loss, accuracy =new_model.evaluate(val_ds)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9950 - loss: 0.0082
Validation Accuracy: 99.50%


In [ ]:
def predict_image(model, img_path, class_names):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)

    # Apply SAME preprocessing as training
    img_array = preprocess_input(img_array)

    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]

    print("Raw prediction:", prediction)

    if prediction < 0.5:
        return class_names[0]   # cats
    else:
        return class_names[1]   # dogs

In [ ]:
class_names = ['cats', 'dogs']
result = predict_image(new_model, "dogsc.jpg", class_names)
print("Prediction:", result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Raw prediction: 0.0015606937
Prediction: cats


In [ ]:
new_model.load_weights("cat_dog_model.keras")

In [ ]:
new_model.save("clean_model.h5", include_optimizer=False)